# Preprocessing

## Setup

Load the tools, find the four files, and set the checking rules. Each five-second piece has 625 rows.


In [1]:
# Clean five-second PPG windows and extract SBP/DBP labels.
# Keep source IDs so recordings can stay together during splitting.
from pathlib import Path
import json
import shutil
import tempfile
from time import perf_counter
import h5py
import numpy as np
from scipy.signal import butter, filtfilt, find_peaks, welch

FS = 125
WINDOW_LEN = FS * 5
FILT_B, FILT_A = butter(4, [0.5 / (FS / 2), 8 / (FS / 2)], btype='band')
MIN_PEAK_DISTANCE = int(FS * 60 / 180)
MIN_BEATS_PER_WINDOW = 3
MIN_INBAND_POWER_RATIO = 0.6
SBP_RANGE = (80, 180)
DBP_RANGE = (40, 110)
PULSE_PRESSURE_RANGE = (20, 100)

DATA_DIR = Path('../data')
DATA_FILES = [DATA_DIR / f'Part_{i}.mat' for i in range(1, 5)]
for path in DATA_FILES:
    if not path.is_file():
        raise FileNotFoundError(path)
run_start = perf_counter()

## Count the pieces and make space

Count the recordings and complete five-second pieces. Use temporary files to avoid filling RAM.

- `X`: PPG readings.
- `y`: SBP and DBP answers.
- Source IDs: where each piece came from.


In [2]:
# Count windows from recording shapes before allocating disk-backed arrays.
candidate_windows = 0
total_recordings = 0
total_samples = 0
for path in DATA_FILES:
    with h5py.File(path, 'r') as f:
        refs = f[path.stem]
        total_recordings += len(refs)
        for ref in refs[:, 0]:
            shape = f[ref].shape
            if len(shape) != 2 or 3 not in shape:
                raise ValueError(f'Unexpected recording shape: {shape}')
            samples = shape[0] if shape[1] == 3 else shape[1]
            total_samples += samples
            candidate_windows += samples // WINDOW_LEN

# Store intermediate arrays on disk to limit RAM use.
temp_dir = Path(tempfile.mkdtemp(prefix='preprocess_', dir=DATA_DIR)).resolve()
if temp_dir.parent != DATA_DIR.resolve():
    raise ValueError('Temporary folder must be inside data/')
X = np.lib.format.open_memmap(temp_dir / 'X.npy', mode='w+', dtype='float32',
                              shape=(candidate_windows, WINDOW_LEN))
y = np.lib.format.open_memmap(temp_dir / 'y.npy', mode='w+', dtype='float64',
                              shape=(candidate_windows, 2))
file_index = np.lib.format.open_memmap(temp_dir / 'file_index.npy', mode='w+',
                                       dtype='int16', shape=(candidate_windows,))
record_index = np.lib.format.open_memmap(temp_dir / 'record_index.npy', mode='w+',
                                         dtype='int32', shape=(candidate_windows,))
start_sample = np.lib.format.open_memmap(temp_dir / 'start_sample.npy', mode='w+',
                                         dtype='int32', shape=(candidate_windows,))
print(f'Recordings: {total_recordings:,}; candidate windows: {candidate_windows:,}')

Recordings: 12,000; candidate windows: 528,828


## Check and clean each piece

- Take **625 rows** at a time to make a five-second window. Leave leftover rows unused.
- Skip windows with missing numbers, flat PPG, or too much noise.
- Find ABP's high and low points. Skip windows with fewer than three of either.
- Calculate the blood-pressure labels:
  - **SBP:** Take the median (middle value) of the ABP high points.
  - **DBP:** Take the median of the ABP low points.
  - The median means sorting the values and taking the middle one. With two middle values, use their average.
- Skip labels outside our chosen blood-pressure limits.
- Clean the PPG:
  - Use a filter to reduce slow drift and fast noise in the signal.
  - Keep the part of the signal between **0.5 and 8 Hz**.
  - The window still has 625 readings; the filter changes their values, not their count.
- Keep the cleaned PPG, its SBP/DBP labels, and its **source IDs**:
  - Which file it came from.
  - Which recording in that file.
  - Which row the window starts at.
  - These let us find the original window and keep the same recording together when splitting data.
- Count how many windows are kept or skipped.


In [3]:
kept = 0
reject_counts = {'non_finite': 0, 'flatline': 0, 'noisy_freq': 0,
                 'too_few_beats': 0, 'bad_bp_range': 0}
file_counts = []

# Open one file at a time and read its recordings.
for file_id, filepath in enumerate(DATA_FILES):
    print(f'Processing {filepath.name}...', flush=True)
    kept_before = kept
    with h5py.File(filepath, 'r') as f:
        record_refs = f[filepath.stem]
        for i in range(len(record_refs)):
            record = f[record_refs[i, 0]][:]
            if record.shape[1] != 3:
                record = record.T
            ppg_full, abp_full = record[:, 0], record[:, 1]

            # Take full five-second windows; leave leftover rows unused.
            for w in range(len(record) // WINDOW_LEN):
                start = w * WINDOW_LEN
                ppg = ppg_full[start:start + WINDOW_LEN]
                abp = abp_full[start:start + WINDOW_LEN]

                # Skip missing or infinite numbers.
                if not np.isfinite(ppg).all() or not np.isfinite(abp).all():
                    reject_counts['non_finite'] += 1
                    continue
                # Skip PPG that barely changes.
                if np.std(ppg) < 1e-3:
                    reject_counts['flatline'] += 1
                    continue

                # Check whether enough PPG signal power is in the chosen frequency range.
                freqs, psd = welch(ppg, fs=FS, nperseg=256)
                inband = (freqs >= 0.5) & (freqs <= 8)
                power_ratio = psd[inband].sum() / psd.sum()
                if not np.isfinite(power_ratio) or power_ratio < MIN_INBAND_POWER_RATIO:
                    reject_counts['noisy_freq'] += 1
                    continue

                # Find ABP high and low points; require at least three of each.
                sys_peaks, _ = find_peaks(abp, distance=MIN_PEAK_DISTANCE)
                dia_peaks, _ = find_peaks(-abp, distance=MIN_PEAK_DISTANCE)
                if len(sys_peaks) < MIN_BEATS_PER_WINDOW or len(dia_peaks) < MIN_BEATS_PER_WINDOW:
                    reject_counts['too_few_beats'] += 1
                    continue

                # Use the median (middle value) of the high and low points as labels.
                sbp = np.median(abp[sys_peaks])
                dbp = np.median(abp[dia_peaks])
                pulse_pressure = sbp - dbp
                # Skip labels outside our chosen blood-pressure limits.
                bp_ok = (SBP_RANGE[0] <= sbp <= SBP_RANGE[1]
                         and DBP_RANGE[0] <= dbp <= DBP_RANGE[1]
                         and PULSE_PRESSURE_RANGE[0] <= pulse_pressure <= PULSE_PRESSURE_RANGE[1])
                if not bp_ok:
                    reject_counts['bad_bp_range'] += 1
                    continue

                # Clean PPG and store the window, labels, and source IDs.
                X[kept] = filtfilt(FILT_B, FILT_A, ppg)
                y[kept] = [sbp, dbp]
                file_index[kept] = file_id
                record_index[kept] = i
                start_sample[kept] = start
                kept += 1
            if (i + 1) % 500 == 0:
                print(f'{filepath.name}: {i + 1}/{len(record_refs)} recordings', flush=True)

    file_counts.append({'file': filepath.name, 'accepted_windows': kept - kept_before})

print(f'Kept {kept:,} windows')
print('Rejected:', reject_counts)

Processing Part_1.mat...


Part_1.mat: 500/3000 recordings


Part_1.mat: 1000/3000 recordings


Part_1.mat: 1500/3000 recordings


Part_1.mat: 2000/3000 recordings


...

Part_4.mat: 1500/3000 recordings


Part_4.mat: 2000/3000 recordings


Part_4.mat: 2500/3000 recordings


Part_4.mat: 3000/3000 recordings


Kept 494,319 windows
Rejected: {'non_finite': 0, 'flatline': 0, 'noisy_freq': 1326, 'too_few_beats': 22, 'bad_bp_range': 33161}


## Check and save

Check that the numbers and counts are valid, then save:

- `processed_dataset.npz`: PPG pieces, blood-pressure answers, and source IDs.
- `data_pipeline_stats.json`: kept/rejected counts and settings.

Remove the temporary files after saving.


In [4]:
# Check the arrays in small batches before saving.
assert kept > 0
assert kept + sum(reject_counts.values()) == candidate_windows
for start in range(0, kept, 2000):
    end = min(start + 2000, kept)
    assert np.isfinite(X[start:end]).all()
    assert np.isfinite(y[start:end]).all()
    assert (y[start:end, 0] > y[start:end, 1]).all()

output_path = DATA_DIR / 'processed_dataset.npz'
np.savez(output_path, X=X[:kept], y=y[:kept],
         file_names=np.array([path.name for path in DATA_FILES]),
         file_index=file_index[:kept], record_index=record_index[:kept],
         start_sample=start_sample[:kept])

stats = {
    'raw_size_bytes': sum(path.stat().st_size for path in DATA_FILES),
    'files_processed': len(DATA_FILES),
    'recordings_processed': total_recordings,
    'signal_samples': total_samples,
    'candidate_windows': candidate_windows,
    'accepted_windows': kept,
    'rejected_windows': sum(reject_counts.values()),
    'rejection_counts': reject_counts,
    'per_file': file_counts,
    'sampling_rate_hz': FS,
    'window_samples': WINDOW_LEN,
    'stride_samples': WINDOW_LEN,
    'label_method': 'median ABP peaks and troughs',
    'bandpass_hz': [0.5, 8],
    'filter_order': 4,
    'min_peak_distance_samples': MIN_PEAK_DISTANCE,
    'min_beats': MIN_BEATS_PER_WINDOW,
    'min_inband_power_ratio': MIN_INBAND_POWER_RATIO,
    'sbp_range': SBP_RANGE,
    'dbp_range': DBP_RANGE,
    'pulse_pressure_range': PULSE_PRESSURE_RANGE,
    'rejection_policy': 'first failing check',
    'runtime_seconds': round(perf_counter() - run_start, 2),
}
Path('../results').mkdir(exist_ok=True)
Path('../results/data_pipeline_stats.json').write_text(json.dumps(stats, indent=2) + '\n')

# Close disk-backed arrays before removing this run's temporary folder.
for array in (X, y, file_index, record_index, start_sample):
    array.flush()
    array._mmap.close()
if temp_dir.parent != DATA_DIR.resolve() or not temp_dir.name.startswith('preprocess_'):
    raise ValueError('Unexpected temporary folder')
shutil.rmtree(temp_dir)
print(f'Saved {output_path} and ../results/data_pipeline_stats.json')
print(f"Runtime: {stats['runtime_seconds']:.2f} seconds")

Saved data\processed_dataset.npz and results/data_pipeline_stats.json
Runtime: 898.64 seconds
